In [53]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv()

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

model=init_chat_model("groq:qwen/qwen3-32b")

model

ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x1544c6f90>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x1544c4f50>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [54]:
from typing import TypedDict, Optional

class AuthState(TypedDict):
    username:Optional[str]
    password:Optional[str]
    is_authenticated:Optional[bool]
    output:Optional[str]


In [55]:
auth_state_1:AuthState={
    "username":"sanskar",
    "password":"sanskar",
    "is_authenticated":True,
    "output":"Login Successful"
}

auth_state_1

{'username': 'sanskar',
 'password': 'sanskar',
 'is_authenticated': True,
 'output': 'Login Successful'}

In [56]:
## will take the input username and password from the user
def input_node(state):
    print(state)

    if(state.get("username","")==""):
        username = input("What is your username?")
    
    password=input("What is your password?")

    if(state.get("username","")==""):
        return {
            "username":username,
            "password":password
        }
    else:
        return {
            "username":state.get("username",""),
            "password":password
        }

In [57]:
result=input_node(auth_state_1)

result

{'username': 'sanskar', 'password': 'sanskar', 'is_authenticated': True, 'output': 'Login Successful'}


{'username': 'sanskar', 'password': 'sanskar'}

In [58]:
## will validate the username and password
def validate_credentials_node(state):
    username=state.get("username","")
    password=state.get("password","")

    print("Username :", username, "Password :", password)

    if(username=="sanskar" and password=="sanskar"):
        is_authenticated=True
    else:
        is_authenticated=False
    
    return {
        "is_authenticated":is_authenticated
    }


In [59]:
validate_credentials(auth_state_1)

Username : sanskar Password : sanskar


{'is_authenticated': True}

In [60]:
# Define the success node
def success_node(state):
    return {"output": "Authentication successful! Welcome."}

In [61]:
# Define the failure node
def failure_node(state):
    return {"output": "Not Successfull, please try again!"}

In [62]:
## router node
def router_node(state):
    print(state)

    if(state["is_authenticated"]):
        return "success_node"
    else:
        return "failure_node"


In [63]:
router_node(auth_state_1)

{'username': 'sanskar', 'password': 'sanskar', 'is_authenticated': True, 'output': 'Login Successful'}


'success_node'

In [64]:
from langgraph.graph import StateGraph
from langgraph.graph import END

# Create an instance of StateGraph with the GraphState structure
workflow = StateGraph(AuthState)
workflow

In [65]:
# nodes
workflow.add_node("InputNode", input_node)
workflow.add_node("ValidateCredential", validate_credentials_node)
workflow.add_node("Success", success_node)
workflow.add_node("Failure", failure_node)

# edges
workflow.add_edge("InputNode", "ValidateCredential")
workflow.add_edge("Success", END)
workflow.add_edge("Failure", "InputNode")
workflow.add_conditional_edges("ValidateCredential", router_node, {"success_node": "Success", "failure_node": "Failure"})

In [66]:
workflow.set_entry_point("InputNode")

In [67]:
app = workflow.compile()


In [68]:
inputs = {"username": "sanskar"}
result = app.invoke(inputs)
print(result)

{'username': 'sanskar'}
Username : sanskar Password : sanskar
{'is_authenticated': True, 'username': 'sanskar', 'password': 'sanskar'}
{'username': 'sanskar', 'password': 'sanskar', 'is_authenticated': True, 'output': 'Authentication successful! Welcome.'}
